In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA

from prophet import Prophet
from xgboost import XGBRegressor

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# =====================================================
# FILES
# =====================================================

TRAIN_FILE = "final_clean_monthly_pm10.csv"
TEST_FILE  = "pm10_2016_2025.csv"

# =====================================================
# LOAD DATA
# =====================================================

train = pd.read_csv(TRAIN_FILE)
test = pd.read_csv(TEST_FILE)

train["Date"] = pd.to_datetime(train["Date"])
test["Date"] = pd.to_datetime(test["Date"])

train = train.set_index("Date")
test = test.set_index("Date")

y_train = train["PM10"]
y_test = test["PM10"]

# =====================================================
# PLOT SERIES
# =====================================================

plt.figure(figsize=(12,5))
plt.plot(y_train)
plt.title("PM10 Training Series")
plt.show()

# =====================================================
# ADF TEST
# =====================================================

print("\nADF TEST")

adf = adfuller(y_train)

print("ADF Statistic =", adf[0])
print("p-value =", adf[1])

if adf[1] > 0.05:
    print("Series is NON-STATIONARY")
    series = y_train.diff().dropna()
    d = 1
else:
    print("Series is STATIONARY")
    series = y_train
    d = 0

# =====================================================
# ACF PACF
# =====================================================

fig,ax = plt.subplots(2,1,figsize=(12,8))

plot_acf(series,lags=48,ax=ax[0])
plot_pacf(series,lags=48,ax=ax[1])

plt.tight_layout()
plt.show()

print("\nUse ACF/PACF plots to identify model:")
print("PACF cutoff -> AR(p)")
print("ACF cutoff  -> MA(q)")
print("Both decay  -> ARMA(p,q)")
print("Seasonal spikes at lag 12 -> SARIMA")

# =====================================================
# CLASSICAL MODELS
# =====================================================

candidate_models = {

    "AR(1)"      : (1,d,0),
    "AR(2)"      : (2,d,0),
    "AR(3)"      : (3,d,0),

    "MA(1)"      : (0,d,1),
    "MA(2)"      : (0,d,2),
    "MA(3)"      : (0,d,3),

    "ARMA(1,1)" : (1,d,1),
    "ARMA(2,1)" : (2,d,1),
    "ARMA(1,2)" : (1,d,2),
    "ARMA(2,2)" : (2,d,2)

}

results = []

predictions = {}

# =====================================================
# FIT CLASSICAL MODELS
# =====================================================

for name,order in candidate_models.items():

    try:

        model = ARIMA(
            y_train,
            order=order
        )

        fit = model.fit()

        pred = fit.forecast(
            len(y_test)
        )

        predictions[name] = pred

        mae = mean_absolute_error(
            y_test,
            pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_test,
                pred
            )
        )

        mape = np.mean(
            np.abs(
                (y_test-pred)/y_test
            )
        )*100

        r2 = r2_score(
            y_test,
            pred
        )

        results.append(
            [name,mae,rmse,mape,r2]
        )

    except:
        pass

# =====================================================
# XGBOOST
# =====================================================

data = train.copy()

for lag in range(1,13):
    data[f"lag_{lag}"] = data["PM10"].shift(lag)

data = data.dropna()

X = data.drop("PM10",axis=1)
y = data["PM10"]

xgb = XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05
)

xgb.fit(X,y)

history = list(y_train)

xgb_pred = []

for i in range(len(y_test)):

    x = np.array(
        history[-12:]
    )[::-1].reshape(1,-1)

    pred = xgb.predict(x)[0]

    xgb_pred.append(pred)

    history.append(pred)

predictions["XGBoost"] = xgb_pred

# =====================================================
# PROPHET
# =====================================================

prophet_train = train.reset_index()
prophet_train.columns = ["ds","y"]

prophet = Prophet(
    yearly_seasonality=True
)

prophet.fit(prophet_train)

future = prophet.make_future_dataframe(
    periods=len(y_test),
    freq="MS"
)

forecast = prophet.predict(future)

prophet_pred = forecast["yhat"].tail(
    len(y_test)
).values

predictions["Prophet"] = prophet_pred

# =====================================================
# LSTM
# =====================================================

scaler = MinMaxScaler()

scaled = scaler.fit_transform(
    train[["PM10"]]
)

lookback = 12

X = []
y = []

for i in range(
    lookback,
    len(scaled)
):

    X.append(
        scaled[i-lookback:i]
    )

    y.append(
        scaled[i]
    )

X = np.array(X)
y = np.array(y)

lstm = Sequential()

lstm.add(
    LSTM(
        64,
        input_shape=(lookback,1)
    )
)

lstm.add(Dense(1))

lstm.compile(
    optimizer="adam",
    loss="mse"
)

lstm.fit(
    X,
    y,
    epochs=100,
    batch_size=16,
    verbose=0
)

window = scaled[-lookback:]

lstm_pred = []

for i in range(len(y_test)):

    pred = lstm.predict(
        window.reshape(1,lookback,1),
        verbose=0
    )

    lstm_pred.append(pred[0,0])

    window = np.vstack(
        [window[1:],pred]
    )

lstm_pred = scaler.inverse_transform(
    np.array(lstm_pred).reshape(-1,1)
).flatten()

predictions["LSTM"] = lstm_pred

# =====================================================
# EVALUATE ML MODELS
# =====================================================

for model_name in ["XGBoost","Prophet","LSTM"]:

    pred = predictions[model_name]

    mae = mean_absolute_error(
        y_test,
        pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            pred
        )
    )

    mape = np.mean(
        np.abs(
            (y_test-pred)/y_test
        )
    )*100

    r2 = r2_score(
        y_test,
        pred
    )

    results.append(
        [model_name,mae,rmse,mape,r2]
    )

# =====================================================
# RESULTS TABLE
# =====================================================

results = pd.DataFrame(
    results,
    columns=[
        "Model",
        "MAE",
        "RMSE",
        "MAPE",
        "R2"
    ]
)

results = results.sort_values(
    "RMSE"
)

print("\nMODEL COMPARISON\n")
print(results)

best_model = results.iloc[0]["Model"]

print("\nBest Model =", best_model)

# =====================================================
# FORECAST PLOT
# =====================================================

plt.figure(figsize=(15,6))

plt.plot(
    y_test.index,
    y_test,
    label="Actual",
    linewidth=3
)

for model in results["Model"][:5]:

    plt.plot(
        y_test.index,
        predictions[model],
        label=model
    )

plt.legend()
plt.title("PM10 Forecast Comparison")
plt.show()

ModuleNotFoundError: No module named 'statsmodels'